# Stage 2 - Chunking

## Data
1_corpus.json

## Chunking strategies

1. 400 chars with overlap 
  - со сдвигом: chunk_size = 40
  - within text items

2. sentence based 
  - within text items 
  - max_tokens=512


Cyrillic text typically tokenizes at 1.8 to 2.5 characters per token (compared to ~4 characters per token in English) because many tokenizers split Russian/Cyrillic words into smaller subwords or byte pairs:
• 772 Cyrillic characters ≈ 310–430 tokens
• Setting a limit below 350 tokens will split most of your average-length documents in half, fragmenting context unnecessarily.

Recommended Configurations by Model Type


Embedding Model Tier	Max Context	Recommended Chunk Size	Overlap	Notes
512-token models (e.g., multilingual-e5-large, rubert-tiny2, sbert_large_nlu_ru)	512 tokens	400–450 tokens	40–50 tokens	Leaves a safe buffer for [CLS] / [SEP] tokens and query prefixes (like "passage: ") without silent truncation.


## Install
pip install tiktoken razdel
pip install sentence_transformers numpy 
pip install tqdm ipywidgets
pip install chromadb

In [10]:
'''
1. Load Docs
'''
import json

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)

print("Разделов:", len(docs))

Разделов: 154


In [11]:
'''
2. Chunking
'''
import tiktoken
from razdel import sentenize  # сегментация русского текста на предложения

ENC = tiktoken.get_encoding("cl100k_base")

def fixed_size_chunks(text, chunk_size=512, overlap=50):
    """Нарезает текст fixed-size-чанкингом: окно из chunk_size токенов со сдвигом chunk_size − overlap."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        window = tokens[start:start + chunk_size]
        chunks.append(ENC.decode(window))
        # Сдвигаем окно: на overlap токенов меньше длины окна
        start += chunk_size - overlap
    return chunks


def sentence_chunks(text, max_tokens=512):
    """Нарезает текст sentence-based-чанкингом: набирает целые предложения, пока не упрётся в бюджет max_tokens."""
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    current = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(ENC.encode(sent))
        # Следующее предложение не влезает — закрываем чанк
        if current and current_tokens + sent_tokens > max_tokens:
            chunks.append(" ".join(current))
            current = []
            current_tokens = 0
        current.append(sent)
        current_tokens += sent_tokens
    if current:
        chunks.append(" ".join(current))
    return chunks


def describe(name, chunks):
    """Печатает сводку по списку чанков: количество, средний размер, текст на стыке соседних чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    avg = sum(sizes) / len(sizes)
    end0 = chunks[0][-24:].replace("\n", " ")
    start1 = chunks[1][:24].replace("\n", " ")
    print(f"{name}")
    print(f"  чанков: {len(chunks)}, средний размер: {avg:.0f}")
    print(f"  размеры, токенов: {sizes}")
    print(f"  чанк 1 кончается на: ...{end0!r}")
    print(f"  чанк 2 начинается с: {start1!r}...")
    print()

In [12]:
chunks_fixed = []
chunks_sentence = []
for i in docs:
  fixed = fixed_size_chunks(i['raw'], chunk_size=400, overlap=40)
  chunks_fixed.extend(fixed)
  
  sentence = sentence_chunks(i['raw'], max_tokens=400)
  chunks_sentence.extend(sentence)

# docs as a list
texts = [item["raw"] for item in docs]
texts_str = ", ".join(texts)

print(f"Корпус: {len(ENC.encode(texts_str))} токена\n")
describe("fixed-size (400/40)", chunks_fixed)
describe("sentence-based (400)", chunks_sentence)

Корпус: 50868 токена

fixed-size (400/40)
  чанков: 224, средний размер: 239
  размеры, токенов: [181, 400, 123, 195, 400, 50, 400, 142, 400, 110, 199, 202, 165, 177, 183, 215, 197, 195, 355, 185, 216, 143, 400, 350, 160, 400, 377, 17, 293, 400, 143, 189, 400, 55, 400, 126, 400, 141, 150, 208, 400, 122, 184, 159, 195, 400, 104, 400, 50, 400, 123, 222, 400, 123, 149, 319, 400, 87, 257, 400, 175, 254, 146, 177, 400, 83, 150, 226, 400, 199, 218, 156, 400, 74, 400, 295, 400, 380, 20, 400, 120, 400, 212, 400, 82, 400, 395, 35, 35, 93, 400, 144, 173, 208, 170, 82, 400, 124, 400, 207, 140, 302, 242, 143, 176, 273, 400, 92, 159, 338, 180, 194, 204, 224, 400, 367, 7, 184, 278, 385, 25, 400, 42, 400, 83, 200, 159, 280, 400, 120, 400, 113, 400, 400, 87, 186, 222, 400, 400, 69, 181, 400, 100, 393, 33, 157, 181, 212, 210, 400, 54, 135, 200, 109, 102, 400, 292, 303, 400, 70, 400, 131, 212, 137, 184, 168, 271, 305, 400, 92, 400, 340, 399, 39, 207, 187, 400, 390, 30, 26, 232, 221, 174, 377, 17, 142, 2

In [13]:
'''
2.1 Chunking Statistics
'''

import statistics  # Медиана и другая базовая статистика

MIN_TOKENS = 100   # Чанк короче этого порога — подозрительно короткий


def is_truncated(chunk):
    """Возвращает True, если чанк не заканчивается завершающим знаком препинания, — значит, оборван."""
    if chunk.rstrip().endswith((".", "!", "?", "…")):
      return False
    else: 
      # print('>>> Truncated: ', chunk)
      return True


def evaluate(name, chunks):
    """Считает метрики качества разбивки: распределение размеров, число коротких и оборванных чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    short = sum(size < MIN_TOKENS for size in sizes)
    truncated = sum(is_truncated(c) for c in chunks)
    print(f"{name}:")
    print(f"  чанков: {len(chunks)}")
    print(f"  размеры, токенов: {sorted(sizes)}")
    print(f"  min / медиана / max: {min(sizes)} / "
          f"{statistics.median(sizes):.0f} / {max(sizes)}")
    print(f"  короче {MIN_TOKENS} токенов: {short}")
    print(f"  обрезаны посреди предложения: {truncated}")

    # for c in chunks:
    #   if len(ENC.encode(c))  < MIN_TOKENS:
    #     print('>>> small size: ', c)
    #    if is_truncated(c):
    #     print('>>> truncated: ', c)


evaluate("fixed-size (400/40)", chunks_fixed)
print()
evaluate("sentence-based (400)", chunks_sentence)

fixed-size (400/40):
  чанков: 224
  размеры, токенов: [7, 17, 17, 20, 22, 25, 26, 30, 33, 35, 35, 39, 40, 42, 45, 50, 50, 54, 55, 69, 70, 73, 74, 82, 82, 83, 83, 87, 87, 90, 92, 92, 93, 97, 100, 102, 104, 109, 110, 113, 120, 120, 122, 123, 123, 123, 124, 124, 126, 131, 131, 135, 137, 140, 141, 142, 142, 143, 143, 143, 144, 146, 149, 150, 150, 156, 157, 159, 159, 159, 160, 161, 163, 164, 165, 168, 170, 173, 174, 174, 175, 176, 177, 177, 177, 180, 181, 181, 181, 183, 184, 184, 184, 185, 186, 187, 189, 194, 195, 195, 195, 197, 199, 199, 199, 200, 200, 202, 203, 204, 207, 207, 208, 208, 210, 212, 212, 212, 215, 215, 216, 218, 221, 222, 222, 224, 226, 232, 240, 242, 254, 257, 271, 273, 278, 278, 279, 280, 284, 292, 293, 295, 297, 299, 302, 303, 305, 319, 338, 340, 340, 350, 354, 355, 367, 377, 377, 380, 382, 385, 390, 393, 395, 399, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400,

In [14]:
'''
2.2 Save to files
'''
chunks_final = []
id = 0
for i in docs:
  chunks = sentence_chunks(i['raw'], max_tokens=400)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_final.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks.json", "w", encoding="utf-8") as f:
  json.dump(chunks_final, f, ensure_ascii=False, indent=4)

In [15]:
'''
3.1 Load chunks for Embedding
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]
metas = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks]

print("Разделов:", len(chunks))

Разделов: 211


In [16]:
'''
3.2 Embed chunks
'''
import numpy as np
from sentence_transformers import SentenceTransformer

def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
corpus_vecs = np.ascontiguousarray(embed_passages(embedder, texts), dtype="float32")
np.save("tmp/kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7223.80it/s]


Чанков: 211, эмбеддинги: (211, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


In [17]:
'''
3.3 Load vectors to DB
'''
import chromadb
from sentence_transformers import SentenceTransformer

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col.add(
    ids=[str(i) for i in range(len(texts))],             # Уникальные id
    embeddings=corpus_vecs.tolist(),                     # Векторы чанков
    documents=texts,                                     # Сами тексты
    metadatas=metas,                                     # Метаданные
)
print("Записей в коллекции:", col.count())

Записей в коллекции: 211


# Stage 2 - Report
Sentence chunking is selected.

2_chunks.json